In [1]:
import numpy as np

In [2]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))
 
def tanh(x):
    return np.tanh(x)
 
def relu(x):
    return np.maximum(x, 0)
 
def lrelu(x, alpha=0.1):
    return np.where(x > 0, x, alpha * x)
 
def prelu(x, a):
    return np.where(x > 0, x, a * x)

def sigmoid_derivative(output):
    return output * (1 - output)
 
def tanh_derivative(output):
    return 1.0 - output ** 2
 
def relu_derivative(output):
    return (output > 0).astype(float)
 
def lrelu_derivative(output, alpha=0.1):
    return np.where(output > 0, 1.0, alpha)
 
def prelu_derivative(output, a):
    return np.where(output > 0, 1.0, a)
 
ACTIVATIONS = {
    'sigmoid': (sigmoid, sigmoid_derivative),
    'relu':    (relu, relu_derivative),
    'tanh':    (tanh, tanh_derivative),
    'lrelu':   (lrelu, lrelu_derivative),
}
 
def mse(output, actual):
    return np.mean((actual - output) ** 2)

In [ ]:
class Layer:
    def __init__(self, num_neurons, num_inputs, activation="sigmoid"):
        # Xavier-ish init, same spirit as your uniform(-1, 1) but scaled
        limit = np.sqrt(6 / (num_inputs + num_neurons))
        self.weights = np.random.uniform(-limit, limit, (num_neurons, num_inputs))
        self.bias = np.zeros(num_neurons)
        self.activation, self.derivative = ACTIVATIONS[activation]

        # stuff for forward / backprop
        self.inputs = None   
        self.total = None    
        self.output = None   
        self.error = None    
 
    def forward(self, inputs):
        self.inputs = inputs
        self.total = self.weights @ inputs + self.bias
        self.output = self.activation(self.total)
        return self.output
 

In [4]:
class NN:
    def __init__(self, num_inputs, hidden_layer_num, hidden_layer_size, num_outputs,
                 hidden_activation="sigmoid", output_activation="sigmoid"):
        self.layers = []
        self.layers.append(Layer(hidden_layer_size, num_inputs, hidden_activation))
        for _ in range(hidden_layer_num - 1):
            self.layers.append(Layer(hidden_layer_size, hidden_layer_size, hidden_activation))
        self.layers.append(Layer(num_outputs, hidden_layer_size, output_activation))
 
    def forward(self, inputs):
        x = np.asarray(inputs, dtype=float)
        for layer in self.layers:
            x = layer.forward(x)
        return x
 
    def backward(self, targets):
        targets = np.asarray(targets, dtype=float)
 
        # output layer error: dL/dz = (output - target) * activation'(output)
        output_layer = self.layers[-1]
        output_layer.error = (output_layer.output - targets) * output_layer.derivative(output_layer.output)
 
        # hidden layers: propagate error backwards using the NEXT layer's weight matrix
        for l in range(len(self.layers) - 2, -1, -1):
            current_layer = self.layers[l]
            next_layer = self.layers[l + 1]
            # next_layer.weights: (next_num_neurons, current_num_neurons)
            # next_layer.error:   (next_num_neurons,)
            propagated = next_layer.weights.T @ next_layer.error
            current_layer.error = propagated * current_layer.derivative(current_layer.output)
 
    def update_weights(self, learning_rate=0.5):
        for layer in self.layers:
            # outer product: (num_neurons, 1) x (1, num_inputs) -> (num_neurons, num_inputs)
            layer.weights -= learning_rate * np.outer(layer.error, layer.inputs)
            layer.bias -= learning_rate * layer.error

In [5]:
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
Y = np.array([[0],    [1],    [1],    [0]],    dtype=float)
 
model = NN(num_inputs=2, hidden_layer_num=1, hidden_layer_size=3, num_outputs=1)

epochs = 20000
learning_rate = 0.5

print("Training neural network...")
for epoch in range(epochs):
    total_loss = 0
    for x, y in zip(X, Y):
        output = model.forward(x)
        total_loss += mse(output, y)
        model.backward(y)
        model.update_weights(learning_rate)

    if epoch % 5000 == 0:
        print(f"Epoch {epoch} - Total Loss: {total_loss:.5f}")

print("\nFinal Predictions after training:")
for x, y in zip(X, Y):
    prediction = model.forward(x)[0]
    print(f"Input: {x} -> Predicted: {prediction:.4f} (Target: {y[0]})")
 

Training neural network...
Epoch 0 - Total Loss: 1.05874
Epoch 5000 - Total Loss: 0.00192
Epoch 10000 - Total Loss: 0.00084
Epoch 15000 - Total Loss: 0.00053

Final Predictions after training:
Input: [0. 0.] -> Predicted: 0.0100 (Target: 0.0)
Input: [0. 1.] -> Predicted: 0.9902 (Target: 1.0)
Input: [1. 0.] -> Predicted: 0.9901 (Target: 1.0)
Input: [1. 1.] -> Predicted: 0.0097 (Target: 0.0)


In [7]:
import torch
import torch.nn as nn


In [8]:
class XORNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(2, 3)   
        self.output = nn.Linear(3, 1)   
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.sigmoid(self.hidden(x))
        x = self.sigmoid(self.output(x))
        return x

X = torch.tensor([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=torch.float32)
Y = torch.tensor([[0],    [1],    [1],    [0]],    dtype=torch.float32)

model = XORNet()
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

for epoch in range(20000):
    optimizer.zero_grad()
    output = model(X)          
    loss = loss_fn(output, Y)
    loss.backward()
    optimizer.step()

    if epoch % 5000 == 0:
        print(f"Epoch {epoch} - Loss: {loss.item():.5f}")

print("\nFinal Predictions:")
with torch.no_grad():
    preds = model(X)
    for x, y, p in zip(X, Y, preds):
        print(f"Input: {x.tolist()} -> Predicted: {p.item():.4f} (Target: {y.item():.1f})")

Epoch 0 - Loss: 0.27077
Epoch 5000 - Loss: 0.01641
Epoch 10000 - Loss: 0.00114
Epoch 15000 - Loss: 0.00056

Final Predictions:
Input: [0.0, 0.0] -> Predicted: 0.0170 (Target: 0.0)
Input: [0.0, 1.0] -> Predicted: 0.9825 (Target: 1.0)
Input: [1.0, 0.0] -> Predicted: 0.9814 (Target: 1.0)
Input: [1.0, 1.0] -> Predicted: 0.0225 (Target: 0.0)
